In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

In [ ]:
data = yf.download(["IBM",'MSFT','INTC'], start="2020-01-01", end="2021-01-01",auto_adjust=False)
stocks = data['Adj Close']#.pct_change().dropna()
stocks

In [ ]:
stocks_returns = (np.log(stocks) - np.log(stocks.shift(1))).dropna()

In [ ]:
stocks_returns_mean = stocks_returns.mean()
weights = np.random.random(len(stocks_returns.columns))
weights /= np.sum(weights)
cov_var = stocks_returns.cov()
port_std = np.sqrt(weights.T.dot(cov_var).dot(weights))

In [ ]:
initial_investment = 1e6
conf_level = 0.95

In [ ]:
from scipy.stats import norm

def VaR_parametric(initial_investment, conf_level):
    alpha = norm.ppf(1 - conf_level, stocks_returns_mean, port_std)
    for i, j in zip(stocks.columns,range(len(stocks.columns))):
        VaR_param = (initial_investment - initial_investment *(1 + alpha))[j]
        print("Parametric VaR result for {} is {} ".format(i, VaR_param))

    VaR_param = (initial_investment - initial_investment * (1 + alpha))
    return VaR_param

In [ ]:
VaR_param = VaR_parametric(initial_investment, conf_level)
VaR_param

In [ ]:
def VaR_historical(initial_investment, conf_level):
    Hist_percentile95 = []
    for i, j in zip(stocks_returns.columns,range(len(stocks_returns.columns))):
        Hist_percentile95.append(np.percentile(stocks_returns.loc[:, i],100*(1-conf_level)))
        print("Based on historical values 95% of {}'s return is {:.4f}".format(i, Hist_percentile95[j]))

    VaR_historical = (initial_investment - initial_investment *(1 + Hist_percentile95[j]))
    print("Historical VaR result for {} is {:.2f} ".format(i, VaR_historical))
   

In [ ]:
VaR_historical(initial_investment,conf_level)

In [ ]:
sim_data = pd.DataFrame([])
num_reps = 1000
n = 100
for i in range(len(stocks.columns)):
    mean = np.random.randn(n).mean()
    std = np.random.randn(n).std()
    temp = pd.DataFrame(np.random.normal(mean, std, num_reps))
    sim_data = pd.concat([sim_data, temp], axis=1)
   
sim_data.columns = ['Simulation 1', 'Simulation 2', 'Simulation 3']



In [ ]:


def MC_VaR(initial_investment, conf_level):
    MC_percentile95 = []

    for i, j in zip(sim_data.columns,range(len(sim_data.columns))):
        MC_percentile95.append(np.percentile(sim_data.loc[:, i], 5))
        print("Based on simulation 95% of {}'s return is {:.4f}".format(i, MC_percentile95[j]))
        VaR_MC = (initial_investment - initial_investment *(1 + MC_percentile95[j]))
        print("Simulation VaR result for {} is {:.2f} ".format(i, VaR_MC))
       

MC_VaR(initial_investment, conf_level)